In [9]:
pip install xlrd

Note: you may need to restart the kernel to use updated packages.


In [10]:
import zipfile
import os

# Path to the ZIP file
zip_path = "MP_LIVE_ALL_BOQ_TDR_WISE.zip"

# Folder where files will be extracted
extract_to = "MP_LIVE_ALL_BOQ_TDR_WISE"

# Extract the ZIP
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print(f"✅ ZIP extracted to: {extract_to}")


✅ ZIP extracted to: MP_LIVE_ALL_BOQ_TDR_WISE


In [16]:
import os
import pandas as pd
import re
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# Clean illegal Excel characters
def clean_illegal_chars(value):
    if isinstance(value, str):
        return ILLEGAL_CHARACTERS_RE.sub("", value)
    return value

# Extract pipe class
def extract_class(text):
    text = str(text).lower()
    if 'k-7' in text:
        return 'K-7'
    elif 'k-9' in text or 'k9' in text:
        return 'K-9'
    elif 'm.s' in text or 'ms' in text:
        return 'M.S'
    elif 'hdpe' in text:
        return 'HDPE'
    elif 'upvc' in text:
        return 'UPVC'
    elif 'pvc' in text:
        return 'PVC'
    elif 'ci' in text:
        return 'CI'
    elif 'di' in text:
        return 'DI'
    else:
        return None

# Extract diameter if valid
def extract_dia(text):
    text = str(text).lower()
    match = re.search(r'(\d+)\s*mm', text)
    if match:
        dia = int(match.group(1))
        if dia in [100, 200, 250, 3500, 1700]:
            return dia
    return None

# Find appropriate column by keywords
def find_column(columns, keywords):
    for col in columns:
        if any(k in str(col).lower() for k in keywords):
            return col
    return None

# Process one BOQ file
def process_boq_file(file_path, output_dir):
    filename = os.path.basename(file_path)
    print(f"🔍 Processing: {filename}")

    try:
        df_raw = pd.read_excel(file_path, header=None)
    except Exception as e:
        print(f"❌ Cannot read {filename}: {e}")
        return

    # Find header row
    header_row_index = None
    for i in range(min(30, len(df_raw))):
        row = df_raw.iloc[i].astype(str).str.lower()
        if any("desc" in cell or "item" in cell for cell in row) and any("unit" in cell for cell in row):
            header_row_index = i
            break

    if header_row_index is None:
        print(f"⚠️ Header not found in {filename}")
        return

    try:
        df = pd.read_excel(file_path, header=header_row_index)
    except Exception as e:
        print(f"❌ Failed to load structured data from {filename}: {e}")
        return

    desc_col = find_column(df.columns, ['desc', 'item'])
    unit_col = find_column(df.columns, ['unit'])
    qty_col = find_column(df.columns, ['qty', 'quantity'])
    rate_col = find_column(df.columns, ['rate', 'amount', 'estimate'])

    if not desc_col or not unit_col:
        print(f"⚠️ Skipping {filename} due to missing columns")
        return

    df[unit_col] = df[unit_col].astype(str).str.strip().str.lower().str.replace('.', '', regex=False)

    def is_pipe_row(row):
        desc = str(row[desc_col]).lower()
        return any(x in desc for x in ["pipe", "di", "ci", "m.s", "k-7", "k-9", "hdpe", "pvc", "upvc"])

    filtered_df = df[df.apply(is_pipe_row, axis=1)].copy()
    if filtered_df.empty:
        print(f"ℹ️ No matching rows in {filename}")
        return

    filtered_df["Item Description"] = filtered_df[desc_col].astype(str)
    filtered_df["class"] = filtered_df["Item Description"].apply(extract_class)
    filtered_df["DIA"] = filtered_df["Item Description"].apply(extract_dia)
    filtered_df["estimate rate"] = pd.to_numeric(filtered_df[rate_col], errors='coerce') if rate_col else 0
    filtered_df["Units"] = df[unit_col].astype(str).str.strip()
    filtered_df["Quantity"] = pd.to_numeric(df[qty_col], errors='coerce') if qty_col else None

    final_df = filtered_df[["Item Description", "class", "DIA", "estimate rate", "Units", "Quantity"]].copy()
    final_df.insert(0, "SL No", range(1, len(final_df) + 1))

    # Clean illegal characters
    final_df = final_df.astype(str).map(clean_illegal_chars)
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, os.path.splitext(filename)[0] + "_filtered.xlsx")
    final_df.to_excel(output_file, index=False, engine="openpyxl")
    print(f"✅ Saved: {output_file}")

# Process all folders and skip already filtered files
def process_all_tdr_folders(main_folder):
    for root, dirs, files in os.walk(main_folder):
        for file in files:
            # Skip temporary, hidden, and already-filtered files
            if file.lower().endswith(('.xls', '.xlsx')) and not file.startswith('~$') and '_filtered' not in file.lower():
                file_path = os.path.join(root, file)
                output_dir = os.path.join(root, "filtered")
                process_boq_file(file_path, output_dir)

# === EXECUTE HERE ===
# Replace this with your extracted folder path:
process_all_tdr_folders("MP_LIVE_ALL_BOQ_TDR_WISE")



🔍 Processing: summary_all.xlsx
⚠️ Header not found in summary_all.xlsx
🔍 Processing: BOQ_348066.xls
⚠️ Header not found in BOQ_348066.xls
🔍 Processing: BOQ_52089.xls
ℹ️ No matching rows in BOQ_52089.xls
🔍 Processing: BOQ_500595.xls
✅ Saved: MP_LIVE_ALL_BOQ_TDR_WISE\MP_LIVE_ALL_BOQ_TDR_WISE\49249167\filtered\BOQ_500595_filtered.xlsx
🔍 Processing: BOQ_500551.xls
✅ Saved: MP_LIVE_ALL_BOQ_TDR_WISE\MP_LIVE_ALL_BOQ_TDR_WISE\49249266\filtered\BOQ_500551_filtered.xlsx
🔍 Processing: BOQ_500499.xls
✅ Saved: MP_LIVE_ALL_BOQ_TDR_WISE\MP_LIVE_ALL_BOQ_TDR_WISE\49249383\filtered\BOQ_500499_filtered.xlsx
🔍 Processing: BOQ_500393.xls
✅ Saved: MP_LIVE_ALL_BOQ_TDR_WISE\MP_LIVE_ALL_BOQ_TDR_WISE\49249507\filtered\BOQ_500393_filtered.xlsx
🔍 Processing: BOQ_500570.xls
✅ Saved: MP_LIVE_ALL_BOQ_TDR_WISE\MP_LIVE_ALL_BOQ_TDR_WISE\49249686\filtered\BOQ_500570_filtered.xlsx
🔍 Processing: BOQ_500671.xls
✅ Saved: MP_LIVE_ALL_BOQ_TDR_WISE\MP_LIVE_ALL_BOQ_TDR_WISE\49250014\filtered\BOQ_500671_filtered.xlsx
🔍 Processin